# 04 — aboutTheJob

Extract job description text from the `aboutTheJob` component response.

Run from repo root.

In [1]:
import json
from urllib.parse import parse_qs, urlparse

from sqlalchemy import text

from core.db import SessionLocal

component_path = "/flagship-web/rsc-action/actions/component"
prefix = "com.linkedin.sdui.generated.jobseeker.dsl.impl."


def component_suffix(url: str) -> str:
    component_id = parse_qs(urlparse(url).query)["componentId"][0]
    return component_id[len(prefix) :] if component_id.startswith(prefix) else component_id


query = text("""
    SELECT request_url, request_body, response_body
    FROM mitm_http_captures
    WHERE request_body IS NOT NULL
      AND response_body IS NOT NULL
    ORDER BY captured_at_ms DESC
""")

with SessionLocal() as session:
    rows = session.execute(query).fetchall()

print(len(rows), "rows loaded")

518 rows loaded


## filter to aboutTheJob

In [2]:
about_the_job_rows = [
    (url, request_body, response_body)
    for url, request_body, response_body in rows
    if urlparse(url).path == component_path
    and component_suffix(url) == "aboutTheJob"
]

print(len(about_the_job_rows), "aboutTheJob rows")
print(about_the_job_rows[0][0])

33 aboutTheJob rows
https://www.linkedin.com/flagship-web/rsc-action/actions/component?componentId=com.linkedin.sdui.generated.jobseeker.dsl.impl.aboutTheJob&sduiid=com.linkedin.sdui.generated.jobseeker.dsl.impl.aboutTheJob&parentSpanId=Yh4SL7pBIZY%3D


## jobId in request body

In [3]:
def job_id_from_request_body(request_body: str) -> str:
    payload = json.loads(request_body)["clientArguments"]["payload"]
    return str(payload["jobId"])


for url, request_body, _ in about_the_job_rows[:5]:
    print(job_id_from_request_body(request_body))

4417153139
4417153139
4419021827
4417153139
4431361538


## pick one jobId

In [4]:
job_id = job_id_from_request_body(about_the_job_rows[0][1])
print(job_id)

matching = [
    (url, request_body, response_body)
    for url, request_body, response_body in about_the_job_rows
    if job_id_from_request_body(request_body) == job_id
]

print(len(matching), "rows for this jobId")
url, request_body, response_body = matching[0]

4417153139
3 rows for this jobId


## chunk ids in this response

In [5]:
lines = response_body.splitlines()
print(len(lines), "lines")

for line in lines[:10]:
    chunk_id, data = line.split(":", 1)
    print(chunk_id, data[:80])

10 lines
1 I["030d6035cb3a997efb1cff7a008d2f89",[],"default"]
3 I["e9e5744c902fddb98f0eb62aee5d400a",[],"TracedComponent"]
4 I["f54a4d9f94904eb227a6c1307124edd6",[],"ClientComponent"]
5 I["b55b61101826bcdf8a734370f7480e4e",[],"VisibleItemsProvider"]
7 I["1e9b95c01e7f142c1ba9a289f4714a9c",[],"default"]
8 "$Sreact.fragment"
9 I["bc12b55b17a9307d0453b2c742c2eca4",[],"default"]
0 ["$","div",null,{"data-sdui-component":"com.linkedin.sdui.generated.jobseeker.ds
6 ["$","$L7",null,{"textProps":{"fontFamily":"sans","fontSize":"small","fontStyle"
2 null


## chunk 6

In [6]:
chunk_6 = None
for line in lines:
    chunk_id, data = line.split(":", 1)
    if chunk_id == "6":
        chunk_6 = json.loads(data)
        break

print(type(chunk_6))
print(chunk_6[:3] if isinstance(chunk_6, list) else chunk_6)

<class 'list'>
['$', '$L7', None]


## textProps tree

In [7]:
_, component_type, key, props_dict = chunk_6
print(component_type, key)
print(props_dict.keys())
text_tree = props_dict["textProps"]["children"]
print(type(text_tree), len(text_tree))

$L7 None
dict_keys(['textProps', 'bindingKey', 'expansionKey', 'expandButtonTextColorExpression', 'maxLineCountExpression', 'onShowMoreAction', 'onShowLessAction', 'textColorExpression', 'viewTrackingSpecs', 'isExpandableTextV2Enabled'])
<class 'list'> 17


## render text from RSC tree

In [8]:
def is_rsc_node(value):
    return isinstance(value, list) and len(value) == 4 and value[0] == "$"


def render_text(element):
    result = []
    if element is None:
        return ""
    if isinstance(element, str):
        if not element.startswith("$"):
            result.append(element)
    elif is_rsc_node(element):
        component_type = element[1]
        if component_type == "strong":
            result.append("\n## ")
        elif component_type == "li":
            result.append("\n- ")
        elif component_type == "br":
            result.append("\n")
        for child in element[-1].get("children", []):
            result.append(render_text(child))
    else:
        for child in element:
            result.append(render_text(child))
    return "".join(result)


description = render_text(text_tree)
print(description[:500])
print("...")
print(len(description), "chars")

HUB24 leads the wealth industry as the best provider of integrated platform, technology and data solutions. At HUB24, we know the smartest investments start with our people. We are innovative and ambitious, and we move fast.

At HUB24, we empower our employees to bring their ideas and creativity to work. Rather than getting bogged down in bureaucracy and red tape, we build a culture that supports our team members to have a real impact on our business and the success of our customers.

HUB24 Limi
...
5528 chars


## extract by jobId

In [9]:
def extract_job_description(job_id: str) -> str | None:
    job_id = str(job_id)
    for url, request_body, response_body in about_the_job_rows:
        if job_id_from_request_body(request_body) != job_id:
            continue
        for line in response_body.splitlines():
            chunk_id, data = line.split(":", 1)
            if chunk_id != "6":
                continue
            _, _, _, props_dict = json.loads(data)
            text_tree = props_dict["textProps"]["children"]
            return render_text(text_tree)
    return None


# try the jobId from above
print(extract_job_description(job_id)[:500])

HUB24 leads the wealth industry as the best provider of integrated platform, technology and data solutions. At HUB24, we know the smartest investments start with our people. We are innovative and ambitious, and we move fast.

At HUB24, we empower our employees to bring their ideas and creativity to work. Rather than getting bogged down in bureaucracy and red tape, we build a culture that supports our team members to have a real impact on our business and the success of our customers.

HUB24 Limi


In [ ]:
# try another jobId from captures
other_job_id = job_id_from_request_body(about_the_job_rows[-1][1])
print("jobId:", other_job_id)
other_description = extract_job_description(other_job_id)
print(other_description[:300] if other_description else None)